# 01 · Audio data: exerciții

**Teorie:** [docs/01_audio_data/READING.md](../../docs/01_audio_data/READING.md)
**Rulează pe:** Mac, gratuit
**Ce exersezi:** încărcare audio, waveform, spectru, spectrogramă, mel, resampling, aliasing, WAV din bytes, streaming.

**Reguli:**
- scrii codul în celulele `# TODO`;
- rulezi celula **✅ Verificare**;
- unde apare ✍️, răspunzi în cuvintele tale.

In [ ]:
# Importuri (nu modifica)
import io
import numpy as np
import librosa, librosa.display
import matplotlib.pyplot as plt
import soundfile as sf

## Exercițiul 1: Încarcă un sunet

Încarcă exemplul `librosa.ex("trumpet")` cu `librosa.load` în `array` și `sr`. Calculează `duration` în secunde **fără** `librosa.get_duration`.

In [ ]:
# TODO
array, sr = None, None
duration = None

In [ ]:
# ✅ Verificare
assert sr == 22050, "librosa.load resamplează implicit la 22050 Hz"
assert array.dtype == np.float32 and np.abs(array).max() <= 1.0, "Audio-ul ar trebui să fie float32 în [-1, 1]"
assert abs(duration - librosa.get_duration(y=array, sr=sr)) < 1e-6
print(f"✅ {len(array)} eșantioane, {duration:.2f} s la {sr} Hz")

✍️ De ce `sr` e 22050 și nu sampling rate-ul original al fișierului? Cum îl păstrezi pe cel original?

## Exercițiul 2: Waveform

Desenează forma de undă cu `librosa.display.waveshow`, pe o figură lată (`plt.figure().set_figwidth(12)`).

In [ ]:
# TODO

✍️ Câte note distingi? Unde e sunetul cel mai puternic?

## Exercițiul 3: Spectrul primelor 4096 de eșantioane

1. Înmulțește `array[:4096]` cu o fereastră `np.hanning(4096)`.
2. Aplică `np.fft.rfft` și ia amplitudinea cu `np.abs`, în `amplitude`. Convertește în dB cu `librosa.amplitude_to_db(..., ref=np.max)`.
3. Calculează frecvențele cu `librosa.fft_frequencies(sr=sr, n_fft=4096)`, în `freqs`.
4. Desenează spectrul cu axa x logaritmică.
5. `peak_hz` este frecvența cu amplitudinea maximă.

In [ ]:
# TODO
amplitude = None
freqs = None
peak_hz = None

In [ ]:
# ✅ Verificare
assert len(freqs) == 4096 // 2 + 1 == len(amplitude)
assert 1200 < peak_hz < 1300, f"peak_hz = {peak_hz:.0f}. Ai luat argmax pe amplitudine și ai indexat freqs?"
print(f"✅ Cel mai puternic vârf: {peak_hz:.0f} Hz")

Cursul spune că **primul** vârf e la ~620 Hz (nota Mi bemol), dar cel mai puternic ți-a ieșit pe la ~1250 Hz.

Calculează `f0_hz`, adică cel mai puternic vârf **sub 1000 Hz**.

In [ ]:
# TODO
f0_hz = None

<details><summary>💡 Indiciu</summary>

Construiește o mască `freqs < 1000` și fă argmax doar pe `amplitude[mask]`.

</details>

In [ ]:
# ✅ Verificare
assert 600 < f0_hz < 650, f"f0_hz = {f0_hz:.0f}"
print(f"✅ Fundamentala ≈ {f0_hz:.0f} Hz; vârful cel mai puternic / fundamentală = {peak_hz / f0_hz:.2f}")

✍️ Ce înseamnă că raportul e ≈ 2? (cuvânt-cheie: armonici)

## Exercițiul 4: Spectrograma (STFT)

1. `D = librosa.stft(array)`, cu parametrii impliciți.
2. `S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)`, apoi `librosa.display.specshow(..., x_axis="time", y_axis="hz")` și `plt.colorbar()`.
3. `hop_ms` este distanța în milisecunde dintre două coloane consecutive ale spectrogramei. `hop_length` implicit este 512.

In [ ]:
# TODO
D = None
hop_ms = None

In [ ]:
# ✅ Verificare
assert D.shape == (1025, 1 + len(array) // 512), f"D.shape = {D.shape}"
assert abs(hop_ms - 512 / sr * 1000) < 0.01
print(f"✅ {D.shape[0]} frecvențe × {D.shape[1]} cadre, câte un cadru la {hop_ms:.1f} ms")

✍️ De ce exact 1025 de frecvențe? (indiciu: `n_fft` implicit)

## Exercițiul 5: Mel spectrogram

Calculează `S = librosa.feature.melspectrogram(...)` cu `n_mels=128` și `fmax=8000`. Convertește-o cu `librosa.power_to_db` și desenează cu `y_axis="mel"`.

In [ ]:
# TODO
S = None

In [ ]:
# ✅ Verificare
assert S.shape == (128, D.shape[1]), f"S.shape = {S.shape}"
print(f"✅ Mel: {S.shape[0]} benzi × {S.shape[1]} cadre")

✍️ De ce `power_to_db` aici și `amplitude_to_db` la STFT? Whisper turbo folosește tot 128 de benzi mel, dar ce sampling rate?

## Exercițiul 6: Resampling la 16 kHz

Resamplează `array` la 16 kHz cu `librosa.resample`, în `y16`.

In [ ]:
# TODO
y16 = None

In [ ]:
# ✅ Verificare
expected = len(array) * 16000 / sr
assert abs(len(y16) - expected) <= 2, f"len(y16) = {len(y16)}, mă așteptam la ~{expected:.0f}"
print(f"✅ {len(array)} → {len(y16)} eșantioane (raport {len(y16)/len(array):.4f} ≈ 16000/22050 = {16000/22050:.4f})")

## Exercițiul 7: Aliasing, sau de ce nu „arunci” eșantioane

1. Generează `tone`: un sinus de **6000 Hz**, 1 secundă, amplitudine 0.5, la `sr = 16000`, `float32`.
2. `naive = tone[::2]` (downsampling „naiv” la 8 kHz) și `proper` = `librosa.resample(tone, orig_sr=16000, target_sr=8000)`.
3. Scrie două funcții:
   - `peak_freq(y, sr)`: frecvența dominantă (refolosește logica de la Ex. 3, cu `np.fft.rfftfreq`);
   - `rms(y)`: rădăcina mediei pătratelor.

In [ ]:
# TODO
tone = None
naive = None
proper = None

def peak_freq(y, sr):
    ...

def rms(y):
    ...

In [ ]:
# ✅ Verificare
assert abs(peak_freq(naive, 8000) - 2000) < 20, "Tonul de 6 kHz ar trebui să apară «fals» la ~2 kHz"
assert rms(proper) < 0.05 * rms(naive), "Resampling-ul corect ar trebui să elimine aproape complet tonul"
print(f"✅ naive: vârf fals la {peak_freq(naive, 8000):.0f} Hz | proper: energie de {rms(naive)/rms(proper):.0f}× mai mică")

✍️ De ce un ton de 6 kHz devine 2 kHz la 8 kHz sampling rate? Ce face `librosa.resample` diferit? (cuvinte-cheie: Nyquist, filtru anti-aliasing)

## Exercițiul 8: WAV din bytes (exact situația din dataset-ul nostru)

În `speechbrain/LoquaciousSet`, audio-ul vine ca **bytes de fișier WAV**. Exersează drumul dus-întors:
1. Scrie `y16` într-un `io.BytesIO` ca WAV, la 16000 Hz, cu `soundfile`. Pune bytes-ii în `wav_bytes`.
2. Decodează-i înapoi cu `sf.read(io.BytesIO(wav_bytes))`, în `y_back` și `sr_back`.

In [ ]:
# TODO
wav_bytes = None
y_back, sr_back = None, None

<details><summary>💡 Indiciu</summary>

`sf.write(buf, y16, 16000, format="WAV")`, apoi `buf.getvalue()`.

</details>

In [ ]:
# ✅ Verificare
assert wav_bytes[:4] == b"RIFF", "Un fișier WAV începe cu b'RIFF'"
assert sr_back == 16000 and len(y_back) == len(y16)
err = np.abs(y_back - y16).max()
assert err < 1e-4, "Diferența ar trebui să fie doar zgomotul de cuantizare"
print(f"✅ {len(wav_bytes)/1024:.0f} KB de WAV; eroare maximă {err:.1e}")

✍️ De ce eroarea e ≈ 3e-5? (indiciu: în ce format scrie `soundfile` un WAV implicit? cât e 1/32768?)

## Exercițiul 9: Streaming

1. Încarcă `hf-internal-testing/librispeech_asr_dummy`, config `"clean"`, split `"validation"`, cu `streaming=True`, în `ds`. Printează `ds.features`.
2. Fă `ds = ds.cast_column("audio", Audio(decode=False))`. Așa primești **bytes-ii** fișierului, nu audio decodat, exact ca în proiect.
3. Primul exemplu merge în `example`. Decodează `example["audio"]["bytes"]` cu `soundfile`, în `y_ex` și `sr_ex`.
4. `first_two` = primele două exemple, cu `.take(2)`.

De ce `decode=False`? Decodarea automată din `datasets` folosește `torchcodec` + FFmpeg. Așa decodăm noi, simplu și controlat, cum vom face în `fetch_audio`. Dacă vrei să încerci și `decode=True`, ai nevoie de un FFmpeg funcțional (`brew reinstall ffmpeg`).

In [ ]:
# TODO
from datasets import load_dataset, Audio
ds = None
example = None
y_ex, sr_ex = None, None
first_two = None

In [ ]:
# ✅ Verificare
assert isinstance(example["audio"]["bytes"], bytes), "Cu decode=False ar trebui să primești bytes"
assert sr_ex == 16000 and len(y_ex) > 0
assert len(first_two) == 2
print(f"✅ {len(y_ex)/sr_ex:.2f} s de audio, primii bytes {example['audio']['bytes'][:4]}, text: {example['text'][:60]}…")

✍️ Bytes-ii încep cu `b'fLaC'`, nu cu `b'RIFF'`. Ce înseamnă? Contează pentru `soundfile`?

## Bonus: vorbire vs trompetă

Desenează mel spectrogram-ul pentru `y_ex`, cu aceleași setări ca la Ex. 5. ✍️ Ce diferențe vezi față de trompetă?

In [ ]:
# TODO (bonus)